In [ ]:
import rasterio
from pyproj import Transformer
import numpy as np
import pandas as pd
from scipy.ndimage import label
import os
from tqdm import tqdm
from scipy.interpolate import RegularGridInterpolator
import time
import geopandas as gpd
from shapely.geometry import Point, box
import xarray as xr
import math
from sklearn.neighbors import KDTree
import matplotlib.pyplot as plt
from eo_tides.model import model_tides

# ================================================
# functions
# ================================================
def calculate_daily_stats(mask, hours_per_day=24):
    """Calculate the average daily inundation duration (hours/day) and the maximum continuous duration"""
    total_days = len(mask) // hours_per_day
    daily_hours = np.zeros(total_days)
    
    for d in range(total_days):
        daily_slice = slice(d * hours_per_day, (d + 1) * hours_per_day)
        daily_hours[d] = np.sum(mask[daily_slice])
    
    avg_daily = np.mean(daily_hours)
    

    labeled, n_components = label(mask)
    if n_components == 0:
        max_consecutive = 0
    else:
        component_sizes = np.bincount(labeled.ravel())[1:]
        max_consecutive = np.max(component_sizes) if len(component_sizes) > 0 else 0
    
    return avg_daily, max_consecutive

def generate_tide_series(lon, lat,times):
    directory=f"D:/tide/tide_models/"
   
    try:
        tide_df = model_tides(
            x=lon,
            y=lat,
            model=["TPXO10-atlas-v2-nc"],
            time=times,
            directory=directory
        )
        return tide_df['tide_height'].values.astype(np.float32)
    except Exception as e:
        print(f"Tide prediction failed: {lon:.4f}E {lat:.4f}N - {str(e)}")
        return None

def save_tide_grid(grid_lons, grid_lats, tide_grid, times, output_dir, format="shp"):
    """Save tidal level"""
    os.makedirs(output_dir, exist_ok=True)
    
    if format.lower() == "shp":
        
        points = []
        attributes = []
    
        max_tide = np.max(tide_grid, axis=0)
        min_tide = np.min(tide_grid, axis=0)
        mean_tide = np.mean(tide_grid, axis=0)
        std_tide = np.std(tide_grid, axis=0)     
     
        for i in range(grid_lons.shape[0]):
            for j in range(grid_lons.shape[1]):
                if np.isnan(mean_tide[i, j]):
                    continue
                    
                point = Point(grid_lons[i, j], grid_lats[i, j])
                points.append(point)
                attributes.append({
                    "grid_id": f"{i}_{j}",
                    "max_tide": max_tide[i, j],
                    "min_tide": min_tide[i, j],
                    "mean_tide": mean_tide[i, j],
                    "std_tide": std_tide[i, j]
                })
        
        if points:
            gdf = gpd.GeoDataFrame(attributes, geometry=points, crs="EPSG:4326")
            output_path = os.path.join(output_dir, f"tide_grid_points_{GRID_ID}.shp")
            gdf.to_file(output_path)
            print(f"Save as Shapefile: {output_path} (Total: {len(points)})")
        else:
            print("No tide saved")
        
    elif format.lower() == "netcdf":
        valid_mask = ~np.isnan(np.mean(tide_grid, axis=0))
        valid_lons = grid_lons[valid_mask]
        valid_lats = grid_lats[valid_mask]
        valid_tide = tide_grid[:, valid_mask]
        
        point_ids = [f"{i}_{j}" for i, j in np.argwhere(valid_mask)]
        
        ds = xr.Dataset(
            {
                "tide": (["time", "point"], valid_tide),
            },
            coords={
                "time": times,
                "lon": (["point"], valid_lons),
                "lat": (["point"], valid_lats),
                "point_id": (["point"], point_ids)
            }
        )
        
        ds.attrs["description"] = "tide data"
        ds.attrs["resolution"] = f"{GRID_RESOLUTION}"
        ds.attrs["model"] = "output"
        ds.attrs["valid_points"] = len(valid_lons)
        
        output_path = os.path.join(output_dir, "tide_grid.nc")
        ds.to_netcdf(output_path)
        print(f"Tide save as NetCDF: {output_path} (Total: {len(valid_lons)})")
    
    else:
        print(f"Wrong format: {format}")

def create_geographic_grid(min_lon, max_lon, min_lat, max_lat, resolution):

    num_lon = math.ceil((max_lon - min_lon) / resolution) + 1
    num_lat = math.ceil((max_lat - min_lat) / resolution) + 1

    lons = np.linspace(min_lon, max_lon, num_lon)
    lats = np.linspace(min_lat, max_lat, num_lat)
    
    lon_grid, lat_grid = np.meshgrid(lons, lats)
    
    return lon_grid, lat_grid, lons, lats

def create_geographic_grid(min_lon, max_lon, min_lat, max_lat, resolution):

    num_lon = math.ceil((max_lon - min_lon) / resolution) + 1
    num_lat = math.ceil((max_lat - min_lat) / resolution) + 1
    

    lons = np.linspace(min_lon, max_lon, num_lon)
    lats = np.linspace(min_lat, max_lat, num_lat)

    lon_grid, lat_grid = np.meshgrid(lons, lats)
    
    return lon_grid, lat_grid, lons, lats

def filter_grid_by_dem_coverage(grid_lons, grid_lats, dem_lons, dem_lats, buffer_deg=0.05):

    dem_points = np.column_stack((dem_lons.ravel(), dem_lats.ravel()))
    dem_tree = KDTree(dem_points)
    
    grid_points = np.column_stack((grid_lons.ravel(), grid_lats.ravel()))

    distances, _ = dem_tree.query(grid_points, k=1)
    distances = distances.reshape(grid_lons.shape)
    
    valid_mask = distances < buffer_deg
    
    num_valid = np.sum(valid_mask)
    num_total = grid_lons.size
    print(f"网格点过滤: 共{num_total}个点, 保留{num_valid}个有效点 ({num_valid/num_total:.1%})")
    
    valid_indices = np.argwhere(valid_mask)
    
    return valid_mask, valid_indices

def plot_grid_coverage(dem_lons, dem_lats, grid_lons, grid_lats, valid_mask, output_dir):
    """绘制网格覆盖图"""
    plt.figure(figsize=(12, 10))
    
    # 绘制DEM点
    plt.scatter(dem_lons, dem_lats, s=1, c='blue', alpha=0.3, label='DEM coverage')
    
    # 绘制所有网格点
    plt.scatter(grid_lons, grid_lats, s=5, c='gray', alpha=0.3, label='All grids')
    
    # 绘制有效网格点
    valid_lons = grid_lons[valid_mask]
    valid_lats = grid_lats[valid_mask]
    plt.scatter(valid_lons, valid_lats, s=10, c='red', alpha=0.7, label='Tide grids')
    
    # 添加标题和标签
    plt.title('Tide modelling area')
    plt.xlabel('Lon')
    plt.ylabel('Lat')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # 保存图像
    output_path = os.path.join(output_dir, f"grid_coverage_{GRID_ID}.png")
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"网格覆盖图已保存: {output_path}")
    plt.close()

def save_tide_cache(cache_dir, grid_lons, grid_lats, tide_grid, times, valid_indices):
    """保存潮位数据到缓存文件"""
    os.makedirs(cache_dir, exist_ok=True)
    cache_file = os.path.join(cache_dir, f"tide_cache_{GRID_ID}.nc")
    
    # 创建xarray数据集
    ds = xr.Dataset(
        {
            "tide": (["time", "y", "x"], tide_grid),
            "valid_mask": (["y", "x"], ~np.isnan(tide_grid[0])),
        },
        coords={
            "time": times,
            "lon": (["y", "x"], grid_lons),
            "lat": (["y", "x"], grid_lats),
        }
    )
    
    # 添加元数据
    ds.attrs["dem_path"] = DEM_PATH
    ds.attrs["start_date"] = START_DATE
    ds.attrs["end_date"] = END_DATE
    ds.attrs["grid_resolution"] = GRID_RESOLUTION
    ds.attrs["buffer_distance"] = BUFFER_DISTANCE
    ds.attrs["valid_indices_count"] = len(valid_indices)
    
    # 保存为NetCDF
    ds.to_netcdf(cache_file)
    print(f"潮位数据已保存到缓存: {cache_file}")
    
    return cache_file

def load_tide_cache(cache_dir):
    """从缓存文件加载潮位数据"""
    cache_file = os.path.join(cache_dir, f"tide_cache_{GRID_ID}.nc")
    
    if not os.path.exists(cache_file):
        return None, None, None, None, None, None
    
    print(f"从缓存加载潮位数据: {cache_file}")
    ds = xr.open_dataset(cache_file)
    
    # 提取数据
    grid_lons = ds["lon"].values
    grid_lats = ds["lat"].values
    tide_grid = ds["tide"].values
    times = ds["time"].values
    valid_mask = ds["valid_mask"].values
    
    # 获取有效点索引
    valid_indices = np.argwhere(valid_mask)
    
    # 验证元数据
    if (
        # ds.attrs["dem_path"] != DEM_PATH or
        ds.attrs["start_date"] != START_DATE or
        ds.attrs["end_date"] != END_DATE or
        ds.attrs["grid_resolution"] != GRID_RESOLUTION or
        ds.attrs["buffer_distance"] != BUFFER_DISTANCE):
        print("警告: 缓存文件配置与当前参数不匹配，将重新计算")
        return None, None, None, None, None
    
    print(f"成功加载潮位数据: {tide_grid.shape} (时间×纬度×经度)")
    return grid_lons, grid_lats, tide_grid, times, valid_indices, valid_mask

def load_dem_data(dem_path):
    """加载DEM数据并计算地理坐标"""
    print("读取DEM数据...")
    with rasterio.open(DEM_PATH) as src:
        dem = src.read(1)
        profile = src.profile.copy()
        transform = src.transform
        crs = src.crs
        rows, cols = dem.shape
        nodata = src.nodata
        print(f"DEM尺寸: {rows}行 × {cols}列, 分辨率: {transform.a:.1f}m")

    # 生成DEM地理坐标（经纬度）
    print("生成DEM地理坐标...")
    dem_xs, dem_ys = np.meshgrid(np.arange(cols), np.arange(rows))
    dem_x_coords, dem_y_coords = transform * (dem_xs, dem_ys)
    
    # 转换到地理坐标系
    transformer_to_geo = Transformer.from_crs(crs, "EPSG:4326", always_xy=True)
    dem_lons, dem_lats = transformer_to_geo.transform(dem_x_coords, dem_y_coords)
    
    # 确定DEM的经纬度范围
    valid_dem_mask = (dem != nodata) & (~np.isnan(dem))
    
    return {
        "dem": dem,
        "profile": profile,
        "transform": transform,
        "crs": crs,
        "rows": rows,
        "cols": cols,
        "nodata": nodata,
        "dem_lons": dem_lons,
        "dem_lats": dem_lats,
        "valid_dem_mask": valid_dem_mask
        }   

def decayed_sum(k, T):
    total = 0.0
    for t in range(1, T+1):  # t从1到T
        term =(1 - k) ** t  # 计算每项 ΔA×(1−k)^t
        total =total+term
    return total
       
# ================================================
# 主处理流程
# ================================================
def main():
    start_time = time.time()
    print(f"开始时间: {time.strftime('%Y-%m-%d %H:%M:%S')}")
    
    # 创建输出目录
    os.makedirs(OUTPUT_DIR, exist_ok=True)


    # 尝试从缓存加载潮位数据
    tide_data_loaded = False
    if USE_TIDE_CACHE:
        grid_lons, grid_lats, tide_grid, times, valid_indices, valid_mask = load_tide_cache(
            TIDE_CACHE_DIR
        )
        if grid_lons is not None:
            tide_data_loaded = True
            grid_rows, grid_cols = grid_lons.shape
            total_hours = len(times)
            total_days = total_hours // 24
  # 加载DEM数据（无论是否使用潮位缓存）
    dem_data = load_dem_data(DEM_PATH)
    dem = dem_data["dem"]
    profile = dem_data["profile"]
    rows = dem_data["rows"]
    cols = dem_data["cols"]
    nodata = dem_data["nodata"]
    dem_lons = dem_data["dem_lons"]
    dem_lats = dem_data["dem_lats"]
    valid_dem_mask = dem_data["valid_dem_mask"]
    # 如果未加载潮位数据，则进行计算
    if not tide_data_loaded:
        valid_dem_lons = dem_lons[valid_dem_mask]
        valid_dem_lats = dem_lats[valid_dem_mask]
    
        min_lon, max_lon = np.min(valid_dem_lons), np.max(valid_dem_lons)
        min_lat, max_lat = np.min(valid_dem_lats), np.max(valid_dem_lats)
        
        # 扩展边界确保覆盖整个DEM
        buffer = BUFFER_DISTANCE
        min_lon -= buffer
        min_lat -= buffer
        max_lon += buffer
        max_lat += buffer
        
        print(f"扩展后范围: {min_lon:.4f}°E - {max_lon:.4f}°E, {min_lat:.4f}°N - {max_lat:.4f}°N")
        
        # 创建地理坐标系网格（经纬度）
        print("创建潮位网格（地理坐标系）...")
        grid_lons, grid_lats, lon_coords, lat_coords = create_geographic_grid(
            min_lon, max_lon, min_lat, max_lat, GRID_RESOLUTION
        )
        
        grid_rows, grid_cols = grid_lons.shape
        print(f"潮位网格尺寸: {grid_rows}行 x {grid_cols}列, 共{grid_rows * grid_cols}个点")
        
        # 过滤网格点：仅保留DEM覆盖范围内的点
        print("过滤网格点（仅保留DEM覆盖范围内的点）...")
        valid_mask, valid_indices = filter_grid_by_dem_coverage(
            grid_lons, grid_lats, valid_dem_lons, valid_dem_lats, BUFFER_DISTANCE
        )
        # 绘制网格覆盖图
        if PLOT_GRID_COVERAGE:
            plot_grid_coverage(valid_dem_lons, valid_dem_lats, grid_lons, grid_lats, valid_mask, OUTPUT_DIR)
        
        # 生成时间序列
        print("生成时间序列...")
        times = pd.date_range(start=START_DATE, end=END_DATE, freq="h")
        total_hours = len(times)
        total_days = total_hours // 24
        print(f"时间序列: {START_DATE} 至 {END_DATE}, 共{total_days}天, {total_hours}小时")
        
        # 计算网格点的潮位时间序列（仅限有效点）
        print("计算有效网格点潮位...")
        tide_grid = np.full((total_hours, grid_rows, grid_cols), np.nan, dtype=np.float32)
        
        # 仅处理有效网格点
        for idx in tqdm(valid_indices, desc="计算潮位网格", unit="点"):
            i, j = idx
            tide_grid[:, i, j] = generate_tide_series(grid_lons[i, j], grid_lats[i, j], times)
        
        # 保存潮位缓存
        if USE_TIDE_CACHE:
            cache_file = save_tide_cache(
                TIDE_CACHE_DIR, 
                grid_lons, grid_lats, tide_grid, times, valid_indices
            )
    
    # 输出潮位网格
    if OUTPUT_TIDE_GRID and not tide_data_loaded:
        print("保存潮位网格数据...")
        save_tide_grid(grid_lons, grid_lats, tide_grid, times, OUTPUT_DIR, TIDE_GRID_FORMAT)
    
    # 创建插值器（地理坐标系）
    print("创建潮位插值器...")
    interpolators = []
    
    # 如果已加载潮位数据，需要重新创建坐标网格
    if tide_data_loaded:
        lon_coords = grid_lons[0, :]
        lat_coords = grid_lats[:, 0]
    
    for t_idx in tqdm(range(total_hours), desc="构建时间插值器", unit="小时"):
        # 创建每个时间步的插值器
        interpolator = RegularGridInterpolator(
            (lat_coords, lon_coords),  # 纬度在前，经度在后
            tide_grid[t_idx, :, :],
            method=INTERPOLATION_METHOD,
            bounds_error=False,
            fill_value=np.nan
        )
        interpolators.append(interpolator)
    
    # 初始化统计矩阵
    stats = {
        "avg_daily_flood": np.full(dem.shape, np.nan, dtype=np.float32),
        # "max_consecutive_flood": np.zeros(dem.shape, dtype=np.int32),
        # "flood_days": np.zeros(dem.shape, dtype=np.int32),
    }
    
    # 创建DEM点坐标数组用于插值
    dem_points = np.column_stack((dem_lats.ravel(), dem_lons.ravel()))
    
    # 逐小时处理
    print("计算淹没状态...")
    flood_mask = np.zeros((total_hours, rows, cols), dtype=bool)
    
    delta_h=A[GRID_ID-1]*decayed_sum(k=k, T=T)*0.01
    print(f"SSP={SSP}; YEAR={YEAR}; GRID={GRID_ID}; A={A[GRID_ID-1]}; delta_h={delta_h}; delta_slr={delta_slr}")
    
    dem=dem+delta_h-S*T-delta_slr
    
    
    for t_idx in tqdm(range(total_hours), desc="处理小时数据", unit="小时"):
        # 获取当前小时的插值器
        interp = interpolators[t_idx]
        
        # 插值得到整个DEM区域的潮位
        tide_values = interp(dem_points).reshape(rows, cols)

        
        # 计算淹没状态（跳过无效高程）
        valid_dem_mask = (dem != nodata) & (~np.isnan(dem))
        

        flood_mask[t_idx] = (tide_values > dem) & valid_dem_mask
    
    # 计算每日统计
    print("计算每日统计...")
    for i in tqdm(range(rows), desc="处理像素", unit="行"):
        for j in range(cols):
            if dem[i, j] == nodata or np.isnan(dem[i, j]):
                continue
            
            # 提取该像素的淹没时间序列
            pixel_flood = flood_mask[:, i, j]
            
            # 计算统计指标
            avg_daily, max_cons = calculate_daily_stats(pixel_flood)
            
            # 计算淹没天数
            flood_days = 0
            for d in range(total_days):
                daily_slice = slice(d * 24, (d + 1) * 24)
                if np.any(pixel_flood[daily_slice]):
                    flood_days += 1
            
            # 存储结果
            stats["avg_daily_flood"][i, j] = avg_daily
            # stats["max_consecutive_flood"][i, j] = max_cons
            # stats["flood_days"][i, j] = flood_days
    
    # 保存GeoTIFF
    print("保存栅格结果...")
    profile.update(dtype=rasterio.float32, nodata=-9999)
    for key in stats:
        output_path = os.path.join(OUTPUT_DIR, f"{key}_{GRID_ID}_{YEAR}.tif")
        with rasterio.open(output_path, "w", **profile) as dst:
            # 对于浮点数据，保留一位小数减少文件大小
            if key == "avg_daily_flood":
                data = np.round(stats[key], 1)
            else:
                data = stats[key]
            dst.write(data.astype(np.float32), 1)
        print(f"已保存: {output_path}")
    
    # 性能统计
    end_time = time.time()
    elapsed = end_time - start_time
    hours, rem = divmod(elapsed, 3600)
    minutes, seconds = divmod(rem, 60)
    print(f"处理完成! 总耗时: {int(hours):02d}h:{int(minutes):02d}m:{seconds:.1f}s")
    
    if not tide_data_loaded:
        print(f"DEM尺寸: {rows}×{cols} = {rows*cols:,}像素")
        print(f"时间步长: {total_hours}小时 ({total_days}天)")
        print(f"潮位网格: {grid_rows}×{grid_cols} = {grid_rows*grid_cols:,}点")
        print(f"有效潮位点: {len(valid_indices):,}点")
    del flood_mask,stats,dem,valid_dem_mask,dem_data,profile

if __name__ == "__main__":
    # ================================================
    # 配置参数
    # ================================================
    for i in range(0, 87):
        BASE_YEAR=2023
        YEAR=2040
        GRID_ID=i
        DEM_PATH = f"F:\\D盘_polyu\\AS\\resample\\{GRID_ID}_dem.tif"         # 输入DEM路径
        OUTPUT_DIR = f"F:\\D盘_polyu\\AS\\resample\\model\\005SSP245\\{YEAR}"           # 输出目录
        START_DATE = "2022-01-01"           # 模拟开始时间
        END_DATE = "2023-12-31 23:00:00"    # 模拟结束时间
        GRID_RESOLUTION = 0.03              # 潮位网格分辨率（度，约3km）
        INTERPOLATION_METHOD = 'nearest'    # 插值方法：'nearest', 'linear'
        OUTPUT_TIDE_GRID = True             # 是否输出潮位网格
        TIDE_GRID_FORMAT = "shp"            # 潮位网格输出格式：shp或netcdf
        BUFFER_DISTANCE = 0.03              # DEM缓冲区距离（度）
        PLOT_GRID_COVERAGE = True           # 是否绘制网格覆盖图
        USE_TIDE_CACHE = True               # 是否使用潮位缓存
        TIDE_CACHE_DIR = r"E:\tide_cache"   # 潮位缓存目录

        scenarios = ['SSP1-2.6', 'SSP2-4.5', 'SSP5-8.5']
        data = {
        'SSP1-2.6': [0.049, 0.092, 0.136, 0.189, 0.232, 0.29, 0.338, 0.386, 0.436, 0.497, 0.545, 0.591, 0.637, 0.681],
        'SSP2-4.5': [0.049, 0.093, 0.142, 0.204, 0.262, 0.33, 0.403, 0.476, 0.556, 0.628, 0.702, 0.775, 0.848, 0.919],
        'SSP5-8.5': [0.05, 0.098, 0.157, 0.232, 0.31, 0.404, 0.509, 0.633, 0.766, 0.856, 0.98, 1.1, 1.214, 1.322]
        }
        years = np.arange(2020, 2160, 10)
        df = pd.DataFrame(data, index=years)
        SSP='SSP2-4.5'
        A=[1.96,1.96,1.96,1.96,0.46,0.46,0.46,0.46,0.46,2.24,
        2.24,2.24,2.24,2.0,2.0,2.0,2.0,2.0,2.0,2.0,
        2.0,2.0,2.33,2.33,2.33,0.85,0.85,0.85,0.85,0.85,
        1.14,1.14,2.3,2.3,2.3,2.3,2.2,2.2,2.2,2.2,
        2.2,4.6,4.6,4.6,2.58,2.58,2.58,1.63,1.63,1.63,
        1.63,3.01,3.01,3.01,1.92,1.92,0.46,0.46,0.4,0.4,
        0.86,0.86,0.51,0.51,3.5,1.24,1.24,1.24,1.24,1.24,
        1.15,1.15,1.15,1.15,1.15,0.87,0.87,0.79,0.79,1.83,
        1.83,1.83,1.83,0.46,0.46,0.46]
        delta_slr=df.loc[YEAR,SSP]-0.049
        k=0.05
        S=0.002
        T=YEAR-BASE_YEAR
        main()